# 02 · Gaussian Actor、联合密度与梯度

## 学习目标

用两维动作看清 `.sum(-1)`；确认单样本 Critic 输出仍是 `[1]`。

默认运行参考实现。练习 Cell 需要主动启用，修改 `.py` 后重新运行。

## 环境准备

先按 [使用指南](README.md) 安装 `.[dev,notebook]`，选择项目 Kernel。

In [1]:
from pathlib import Path
import os
import sys
import torch
import numpy as np

# 从仓库根目录或 notebooks/ 打开都可；不要依赖某台机器的绝对路径。
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'pyproject.toml').exists()
                     and (p / 'src/ppo_pendulum').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('请从项目目录启动 JupyterLab')
os.chdir(PROJECT_ROOT)
%load_ext autoreload
%autoreload 2
torch.manual_seed(7)
np.random.seed(7)
print('Python:', sys.executable)
print('Project:', PROJECT_ROOT.name)

Python: /Users/matt/Documents/Codex/2026-11-15/referenced-chatgpt-conversation-this-is-an-2/work/.venv/bin/python
Project: ppo-inverted-pendulum


## 1. 构造两维动作例子

这里暂时用 `action_dim=2`，更容易观察联合 log density。实际倒立摆是 1 维动作。

In [2]:
from ppo_pendulum.reference.model import GaussianActorCritic
model = GaussianActorCritic(obs_dim=4, action_dim=2)
observations = torch.zeros(3, 4)
dist = model.distribution(observations)
actions = dist.sample()
per_dim_log_prob = dist.log_prob(actions)
joint_log_prob = model.joint_log_prob(dist, actions)
print('actions:', actions)
print('per dimension:', per_dim_log_prob)
print('joint:', joint_log_prob)
assert per_dim_log_prob.shape == (3, 2)
assert joint_log_prob.shape == (3,)
torch.testing.assert_close(joint_log_prob, per_dim_log_prob.sum(-1))
assert model.value(torch.zeros(1, 4)).shape == (1,)

actions: tensor([[ 0.6370,  0.2429],
        [-0.2284, -0.0382],
        [-0.3219, -1.4324]])
per dimension: tensor([[-0.9705, -0.4991],
        [-0.4898, -0.4209],
        [-0.5598, -3.2076]], grad_fn=<SubBackward0>)
joint: tensor([-1.4696, -0.9107, -3.7674], grad_fn=<SumBackward1>)


## 2. 同一组动作与固定优势的策略梯度

`actions` 和 `advantages` 是已采集数据；梯度流经当前策略的 log probability。共享一个优势并不意味着各动作维的参数梯度相同。

In [3]:
advantages = torch.tensor([1.0, -0.5, 0.25])
loss = -(joint_log_prob * advantages).mean()
model.zero_grad()
loss.backward()
print('Actor loss:', loss.item())
print('log_std gradient:', model.log_std.grad)
assert torch.isfinite(model.log_std.grad).all()
assert all(p.grad is None for p in model.critic_parameters())

Actor loss: 0.6520242094993591
log_std gradient: tensor([-0.1175, -0.2676])


## 3. 手写 TODO

编辑 `src/ppo_pendulum/exercises/actor_critic.py`，实现 `joint_log_prob` 与 `Critic.forward`，保存后启用下方检查。

In [4]:
RUN_EXERCISE = False
if RUN_EXERCISE:
    import subprocess
    result = subprocess.run([sys.executable, '-m', 'pytest', '-q',
                             'checks/test_exercises.py', '-k', 'stage_2'], check=True)
else:
    print('练习检查尚未启用；完成 TODO 后设 RUN_EXERCISE=True。')

练习检查尚未启用；完成 TODO 后设 RUN_EXERCISE=True。


## 检查

改变 batch 大小和动作维度。联合 log probability 应始终是 `[B]`；不要把 batch 维求和，也不要用无参数 `.squeeze()`。

## 下一步

打开 `03_gae_and_bootstrap.ipynb`，学习 Actor 的优势和 Critic 的 target 从哪里来。

公式对照：[PPO_FORMULAS.md](../docs/PPO_FORMULAS.md)。